# ARIA CrowdSim (Colab)

Public instrument field + public map `π(r, Z)` + Phase A cover + critical-value Phase B.

**Authors:** Seyed Ali Hosseini, Omid Sojodishijani, Vahid Khajehvand

This notebook is the closed-form artifact. No sequence model is trained. Worlds are paired by seed.

If you also uploaded the project folder, skip the bootstrap cell and do:
`import sys; sys.path.append('/content/ARIA_complete')`

In [ ]:
!pip -q install numpy pandas matplotlib
import numpy as np, pandas as pd, time, pathlib, sys

## Option A — use the project package
Upload `ARIA_complete.zip`, unzip, then run the next cell.

In [ ]:
from pathlib import Path
root = Path('/content/ARIA_complete')
if root.exists():
    sys.path.insert(0, str(root))
    print('package ready', root)
else:
    print('package not found; use Option B cells below or unzip ARIA_complete.zip')

In [ ]:
try:
    from aria.config import SimConfig
    from aria.engine import run_selector
    HAVE_PKG = True
except Exception as e:
    HAVE_PKG = False
    print('fallback path', e)

## Smoke benchmark (paired seeds)
Default is small so Colab finishes in about a minute. For paper scale set `PAPER = True`.

In [ ]:
PAPER = False
n, t, seeds, task_mean = (500, 24, 10, 80.0) if PAPER else (60, 6, 2, 20.0)
selectors = ['posted','gtdim_if','locked_cpt','cover_only','field_only','aria_private','aria_public']
rows = []
if HAVE_PKG:
    for seed in range(seeds):
        for name in selectors:
            cfg = SimConfig(n=n, t_slots=t, task_mean=task_mean, seed=seed)
            t0 = time.perf_counter()
            ep = run_selector(name, cfg)
            s = ep.summary()
            s.update(selector=name, seed=seed, ms=1000*(time.perf_counter()-t0)/t)
            rows.append(s)
    df = pd.DataFrame(rows)
    display(df.groupby('selector')[['W_P','TCR','ICR','IR_v']].mean().round(2))
else:
    print('Install / unzip the package first.')

## Attack check
Public `π(r,Z)` must give `Δx = 0` by construction. Private map is movable.

In [ ]:
if HAVE_PKG:
    from aria.field import PublicMap, PrivateMap
    from aria.world import CrowdWorld
    cfg = SimConfig(n=80, t_slots=1, seed=0, attack=False)
    w0 = CrowdWorld(cfg).sample_slot(0)
    cfg_a = SimConfig(n=80, t_slots=1, seed=0, attack=True)
    w1 = CrowdWorld(cfg_a).sample_slot(0)
    r = np.array([1.0, 0.3, 0.1, 0.2, 0.1])
    pub, priv = PublicMap(), PrivateMap()
    dx_pub = np.abs(pub(r, 0.0) - pub(r, 0.0)).sum()
    dx_priv = np.abs(priv(r, 0.0, w0.encoding) - priv(r, 0.0, w1.encoding)).sum()
    print('public L1 displacement (must be 0):', float(dx_pub))
    print('private L1 displacement (should be > 0):', float(dx_priv))